In [ ]:
import pickle, mesh, wall_generation, visualization, numpy as np, parametrization
from tri_mesh_viewer import TriMeshViewer, TextureMap, FlatteningAnimation
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf

In [ ]:
surf = mesh.paraboloid(k1=1, k2=1, triArea=0.0005)

In [ ]:
viewer = TriMeshViewer(surf, width=768, height=512)
viewer.showWireframe()
viewer.show()

In [ ]:
# Choose reasonable stretching bounds
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
lg = parametrization.LocalGlobalParametrizer(surf, parametrization.lscm(surf))

for i in range(1000): lg.runIteration()
print(lg.energy())
lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

print(lg.energy())
lg.runIteration()
print(lg.energy())

rparam = parametrization.RegularizedParametrizerSVD(lg)
for i in range(5000): lg.runIteration()
print(lg.energy())

In [ ]:
#rparam = parametrization.RegularizedParametrizerSVD(lilium, np.loadtxt('data/lilium_tower_parametrization_4_10_2019.txt'))
rparam = parametrization.RegularizedParametrizerSVD(surf, lg.uv())
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = 2000
    opts.gradTol = 1e-10
    parametrization.benchmark_reset()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)
    parametrization.benchmark_report()

In [ ]:
with suppress_stdout(): optimize_rparam(rparam, 2e-3, 1e-4)
with suppress_stdout(): optimize_rparam(rparam, 1e-4, 3e-5)
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 2e-5)
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5)
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-6)

In [ ]:
with suppress_stdout(): optimize_rparam(rparam, 0, 0)

In [ ]:
PET = parametrization.RegularizedParametrizerSVD.EnergyType
list(map(rparam.energy, [PET.Fitting, PET.AlphaRegularization, PET.PhiRegularization]))

In [ ]:
visualization.visualize(rparam)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False)

In [ ]:
visualization.singularValueHistogram(rparam)

In [ ]:
nsubdiv=2
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=80)

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, height=12)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.02,
                                              minContourLen=0.075)

In [ ]:
visualization.plot_line_segments(pts, edges, width=20, height=16)

## Meshing and inflation simulation

In [ ]:
m, fuseMarkers = wall_generation.triangulate_channel_walls(pts[:,0:2], edges, 0.0005)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=20, height=18)

In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet.visualizationMesh(), width=1024, height=768)
viewer.showWireframe()
viewer.show()

In [ ]:
isheet.setUseTensionFieldEnergy(True)

In [ ]:
import time
isheet.pressure = 10
inflation.benchmark_reset()
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)
    if cr.numIters() < iterations_per_output: break
    #isheet.writeDebugMesh('lilium_inflate/inflation_tf_lifted_step_{}.msh'.format(step))
    viewer.update(False, isheet.visualizationMesh())
    time.sleep(0.05) # Allow some mesh synchronization time for pythreejs
inflation.benchmark_report()

In [ ]:
isheet.tensionStateHistogram()